# RuralCare AI — Multi-Agent Rural Healthcare Assistant

**Kaggle Demo Notebook**

> ⚠️ **IMPORTANT DISCLAIMER:** RuralCare AI is NOT a doctor and cannot diagnose illness or prescribe medicines.  
> All outputs are for general health awareness only.  
> Always consult a qualified healthcare professional.  
> **In an emergency, call 112 (Emergency) or 108 (Ambulance) immediately.**

---

Demonstrates a multilingual, RAG-powered multi-agent AI system for rural healthcare triage across 5 representative cases.


## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'langchain', 'langchain-anthropic', 'langchain-community',
    'langgraph>=0.2', 'chromadb', 'sentence-transformers',
    'python-dotenv', 'pydantic>=2'
], check=True)
print('Dependencies installed')

In [ ]:
import os, sys
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['ANTHROPIC_API_KEY'] = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
    print('API key loaded from Kaggle Secrets')
except Exception:
    os.environ.setdefault('ANTHROPIC_API_KEY', 'YOUR_API_KEY_HERE')
    print('Set ANTHROPIC_API_KEY if you see auth errors')

PROJECT_ROOT = Path('/kaggle/working/RuralCare_AI_Claude')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('LLM_PROVIDER', 'claude')
os.environ.setdefault('DEMO_MODE', 'true')
os.environ.setdefault('CHROMA_DB_PATH', str(PROJECT_ROOT / 'data' / 'chroma'))
os.environ.setdefault('DATABASE_URL', f'sqlite:///{PROJECT_ROOT}/data/ruralcare.db')
os.environ.setdefault('LOG_LEVEL', 'WARNING')
print(f'Project root: {PROJECT_ROOT}')

## 2. Initialise Database & Knowledge Base

In [ ]:
from app.database.sqlite_client import init_db
from app.rag.vector_store import init_vector_store
from app.rag.document_loader import ingest_demo_docs

init_db()
init_vector_store()
ingest_demo_docs()
print('Database, vector store, and demo health guidelines ready.')

## 3. Helper — Pretty-print Results

In [ ]:
TRIAGE_EMOJI = {'EMERGENCY': '🔴', 'URGENT': '🟠', 'MODERATE': '🟡', 'MILD': '🟢'}

def display_result(case_name, result):
    level = result.get('triage_level', 'UNKNOWN')
    emoji = TRIAGE_EMOJI.get(level, '⚪')
    print('=' * 65)
    print(f' {case_name}')
    print('=' * 65)
    print(f' Triage : {emoji} {level}')
    print(f' Safety : {"Passed" if result.get("safety_passed") else "Filtered"}')
    if result.get('emergency_flag'):
        print(f' EMERGENCY ALERT: {result.get("emergency_alert", "")[:200]}')
    guidance = result.get('health_guidance') or result.get('final_response') or ''
    print(f' Guidance: {guidance[:300]}')
    if result.get('recommended_facility'):
        print(f' Facility: {result["recommended_facility"]}')
    print(f' Session: {result.get("session_id", "")[:20]}...')

## 4. Five Demo Cases

In [ ]:
from app.services.orchestrator import run_pipeline

# Case 1 — Mild
r1 = run_pipeline(
    raw_input='I have a mild headache and feel a bit tired since this morning.',
    language='en',
    location_district='Dharmapuri',
    location_state='Tamil Nadu',
)
display_result('CASE 1 — Mild: Headache (English)', r1)

In [ ]:
# Case 2 — Moderate
r2 = run_pipeline(
    raw_input='I have had fever for 3 days, headache, and body aches. I feel very weak.',
    language='en',
    location_district='Dharmapuri',
    location_state='Tamil Nadu',
)
display_result('CASE 2 — Moderate: Fever 3 days (English)', r2)

In [ ]:
# Case 3 — Urgent
r3 = run_pipeline(
    raw_input='I have chest pain and have been coughing for 2 weeks. I am also losing weight.',
    language='en',
    location_district='Salem',
    location_state='Tamil Nadu',
)
display_result('CASE 3 — Urgent: Chest Pain + Cough (English)', r3)

In [ ]:
# Case 4 — Emergency (fast-path — NO LLM for triage)
r4 = run_pipeline(
    raw_input='My father is unconscious and cannot breathe. Please help.',
    language='en',
)
display_result('CASE 4 — EMERGENCY: Unconscious (English)', r4)
print(f'Emergency fast-path triggered: {r4.get("emergency_flag")}')

In [ ]:
# Case 5 — Multilingual (Hindi)
r5 = run_pipeline(
    raw_input='मुझे तीन दिनों से बुखार है और सिर दर्द हो रहा है।',
    language='hi',
    location_district='Varanasi',
    location_state='Uttar Pradesh',
)
display_result('CASE 5 — Multilingual: Hindi Fever Query', r5)

## 5. Safety Filter Verification

In [ ]:
from app.utils.safety_filter import detect_emergency, run_safety_filter

EMERGENCY_TESTS = [
    ('chest pain and cannot breathe', True),
    ('I feel a bit tired today',       False),
    ('unconscious on the floor',       True),
    ('mild headache since morning',    False),
]
print('Emergency keyword detection:')
for text, expected in EMERGENCY_TESTS:
    got = detect_emergency(text)
    print(f'  {"PASS" if got == expected else "FAIL"}: "{text}" -> {got}')

def _check_filter(guidance, should_block):
    state = {
        'health_guidance': guidance, 'triage_level': 'MODERATE',
        'emergency_flag': False, 'patient_token': 'PT-nb001',
        'session_id': 'nb-safety', 'audit_log': [],
    }
    r = run_safety_filter(state)
    blocked = not r.get('safety_passed', True)
    print(f'  {"PASS" if blocked == should_block else "FAIL"}: [{guidance[:50]}...] blocked={blocked}')

print('\nSafety filter — SHOULD block:')
_check_filter('You have malaria and need immediate treatment.', True)
_check_filter('Take paracetamol 500mg twice daily for 5 days.', True)

print('\nSafety filter — SHOULD pass:')
_check_filter('Please visit the nearest PHC for evaluation.', False)
_check_filter('Stay hydrated and rest. See a doctor if symptoms worsen.', False)

## 6. Audit Log

In [ ]:
from app.database.sqlite_client import get_recent_audit_logs
import pandas as pd

logs = get_recent_audit_logs(limit=50)
if logs:
    df = pd.DataFrame(logs)
    show = [c for c in ['agent_name','triage_level','emergency_flag','safety_passed','blocked_reason','created_at'] if c in df.columns]
    print(f'Total audit entries: {len(df)}')
    print(df[show].tail(15).to_string())
else:
    print('No audit entries yet.')

## 7. Summary

| Case | Input | Expected Triage | Emergency Fast-path |
|------|-------|----------------|--------------------|
| 1 | Mild headache (EN) | MILD | No |
| 2 | Fever 3 days (EN) | MODERATE | No |
| 3 | Chest pain + cough (EN) | URGENT | No |
| 4 | Unconscious (EN) | EMERGENCY | **YES** |
| 5 | Fever in Hindi | MODERATE | No |

**Key Safety Properties:**
- Emergency keywords detected before any LLM call (rule-based <100ms)
- Safety filter blocks diagnosis language and specific medication dosages
- Disclaimer always injected on every patient-facing response
- Audit log always written — zero skips
- PHI anonymised — patient tokens used in all LLM prompts

---
*RuralCare AI v0.1.0 — Demo Only — Not for Clinical Use*
